# Simulation 1 - CRT - KDE + ECDF - Figures
This notebook takes outputs from the Simulation 1 KDE and ECDF experiments and regenerates the cleaned manuscript figures. To run, change the input paths to the output of the desired run. 

Dependencies: json, numpy, matplotlib, scipy

### ECDF - Slope Plots

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm


# ============================================================
# Update sizing
# ============================================================
import matplotlib as mpl 
mpl.rcParams["font.sans-serif"] = [
    "Arial"
]

# Huge font 
mpl.rcParams["font.size"] = 12
mpl.rcParams["axes.titlesize"] = 26
mpl.rcParams["axes.labelsize"] = 26
mpl.rcParams["legend.fontsize"] = 17 
mpl.rcParams["legend.title_fontsize"] = 16
mpl.rcParams["xtick.labelsize"] = 24
mpl.rcParams["ytick.labelsize"] = 24

# Cleaner editable text in PDF/SVG
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"


# ============================================================
# Point this to the saved run folder
# ============================================================
RUN_DIR = Path("logs/ECDF/alpha_h0=0.5_T=2000_m1=50_nreps=100_2026-05-19-14:25")
RESULTS_PATH = RUN_DIR / "all_results.json"

REMAKE_FIG_DIR = RUN_DIR / "remade_figures"
REMAKE_FIG_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# Reconstruct the true density grid from original constants
# ============================================================
w1 = 0.35
mu1, sigma1 = -2.0, 0.8
mu2, sigma2 = 1.0, 1.3

left = min(mu1 - 6 * sigma1, mu2 - 6 * sigma2)
right = max(mu1 + 6 * sigma1, mu2 + 6 * sigma2)
m_grid = 500

xg = np.linspace(left, right, m_grid)

f_true_g = (
    w1 * norm.pdf(xg, mu1, sigma1)
    + (1.0 - w1) * norm.pdf(xg, mu2, sigma2)
)


# ============================================================
# Read saved JSON
# ============================================================
with open(RESULTS_PATH, "r") as f:
    all_results = json.load(f)

print(f"Loaded {len(all_results)} saved result entries.")


# ============================================================
# Plot helpers
# ============================================================
def plot_density_from_saved(res):
    """
    Remake density plot using the final saved f_hat_history
    from the saved result entry.
    
    Note: in your current script, all_results stores last_out,
    so this is the density from the LAST repetition, not the first repetition.
    """
    p = float(res["p_target"])
    alpha = float(res["alpha"])

    f_hat_history = np.asarray(res["f_hat_history"])
    f_hat_final = f_hat_history[-1]

    plt.figure(figsize=(7, 4))
    plt.plot(xg, f_true_g, linewidth=2, label="True density")
    plt.plot(xg, f_hat_final, linewidth=2, label="Estim. density")
    plt.xlabel("x")
    plt.ylabel("density")
    # plt.title(rf"Estimated density: p={p:.2f}, $\alpha$={alpha:.2f}")
    plt.grid(True, axis="y", ls="--", alpha=0.3)
    plt.legend()
    plt.tight_layout()

    out_path = REMAKE_FIG_DIR / f"density_saved_p={p:.2f}_alpha={alpha:.2f}.pdf"
    plt.savefig(out_path, bbox_inches="tight")
    plt.close()

    print(f"Saved {out_path}")


def plot_with_trend(M, values, metric_name, p_val, alpha_val, drop_first=30, save_path=None):
    eps = 1e-20
    M = np.asarray(M, dtype=float)
    values = np.asarray(values, dtype=float)

    log_M = np.log10(M)
    log_val = np.log10(values + eps)

    plt.figure(figsize=(6, 4))
    plt.plot(log_M, log_val, "o-", alpha=0.5, markersize=3, label="Distr. Loss")

    if len(M) > drop_first + 5:
        x_fit = log_M[drop_first:]
        y_fit = log_val[drop_first:]
        slope, intercept = np.polyfit(x_fit, y_fit, 1)
        y_pred = slope * x_fit + intercept
        plt.plot(x_fit, y_pred, "r--", linewidth=2, label=f"Fit slope: {slope:.3f}")

    plt.xlabel(r"$\log_{10}$(M$_n$)")

    if metric_name == "W1":
        plt.ylabel(r"Mean Slope $W_1$")
        # plt.title(r"Mean $W_1$ Slope vs $\alpha$")
    elif metric_name == 'MMD':
        plt.ylabel(f"Mean Slope {metric_name}")
        # plt.title(rf"Mean {metric_label} Slope vs $\alpha$")
    else: 
        pass # no label 

    plt.legend()
    plt.grid(True, which="both", ls="--", alpha=0.3)
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.close()
        print(f"Saved {save_path}")
    else:
        plt.show()


def remake_scaling_plots(res):
    p = float(res["p_target"])
    alpha = float(res["alpha"])

    M = np.asarray(res["M"], dtype=float)
    W1 = np.asarray(res["W1"], dtype=float)
    MMD2 = np.asarray(res["MMD2"], dtype=float)
    MMD = np.sqrt(np.maximum(MMD2, 0.0))

    run_label = f"p={p:.2f}_alpha={alpha:.2f}"

    plot_with_trend(
        M,
        W1,
        "W1",
        p,
        alpha,
        drop_first=30,
        save_path=REMAKE_FIG_DIR / f"{run_label}_W1.pdf",
    )

    plot_with_trend(
        M,
        MMD,
        "MMD",
        p,
        alpha,
        drop_first=30,
        save_path=REMAKE_FIG_DIR / f"{run_label}_MMD.pdf",
    )

    plot_with_trend(
        M,
        MMD2,
        r"$\text{MMD}^2$",
        p,
        alpha,
        drop_first=30,
        save_path=REMAKE_FIG_DIR / f"{run_label}_MMD2.pdf",
    )


def plot_alpha_vs_mean_slope(metric_key, slopes_key, metric_label, filename, use_error_bars=False):
    unique_ps = sorted(set(float(res["p_target"]) for res in all_results))

    plt.figure(figsize=(6, 4))
    DARK_BLUE = "#244B99"
    RED = "#CC0000"

    for p in unique_ps:
        subset = [r for r in all_results if float(r["p_target"]) == p]

        alpha_plot = np.array([float(r["alpha"]) for r in subset])
        means = np.array([float(r[metric_key]) for r in subset])

        order = np.argsort(alpha_plot)
        alpha_plot = alpha_plot[order]
        means = means[order]

        vals_plot = -means

        if use_error_bars:
            stds = np.array([
                np.asarray(r[slopes_key], dtype=float).std(ddof=1)
                for r in subset
            ])
            stds = stds[order]

            n_reps = len(subset[0][slopes_key])
            err_plot = 1.96 * stds / np.sqrt(n_reps)

            plt.errorbar(
                alpha_plot,
                vals_plot,
                yerr=err_plot,
                fmt="o-",
                color=DARK_BLUE,
                ecolor=DARK_BLUE,
                elinewidth=1,
                capsize=3,
                label="Empirical",
            )
        else:
            plt.plot(alpha_plot, vals_plot, "o-", color=DARK_BLUE, label="Empirical")

        theo_vals = np.minimum(p, alpha_plot)
        plt.plot(alpha_plot, theo_vals, "--", color=RED, label="Theoretical")

    plt.xlabel(r"$\alpha$")

    if metric_label == "W1":
        plt.ylabel(r"Mean Slope $W_1$")
        # plt.title(r"Mean $W_1$ Slope vs $\alpha$")
    elif metric_label == "W1":
        plt.ylabel(f"Mean Slope {metric_label}")
        # plt.title(rf"Mean {metric_label} Slope vs $\alpha$")
    else:
        pass

    plt.grid(True, axis="y", ls="--", alpha=0.3)
    plt.ylim(0.0, 0.65)
    plt.yticks(np.arange(0.0, 0.7, 0.1))

    handles, labels = plt.gca().get_legend_handles_labels()
    uniq = dict(zip(labels, handles))
    legend=False
    if legend:
        plt.legend(uniq.values(), uniq.keys())

    plt.tight_layout()
    out_path = REMAKE_FIG_DIR / filename
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()

    print(f"Saved {out_path}")


# ============================================================
# Remake all per-alpha plots
# ============================================================
for res in all_results:
    remake_scaling_plots(res)
    plot_density_from_saved(res)



## ECDF - Summary Plots

In [ ]:
## ECDF Summary Plots 

# ============================================================
# Update sizing
# ============================================================
import matplotlib as mpl 
mpl.rcParams["font.sans-serif"] = [
    "Arial",
]

# Smaller font 
mpl.rcParams["font.size"] = 12
mpl.rcParams["axes.titlesize"] = 18
mpl.rcParams["axes.labelsize"] = 18
mpl.rcParams["legend.fontsize"] = 11
mpl.rcParams["legend.title_fontsize"] = 14
# Axis tick label sizes
mpl.rcParams["xtick.labelsize"] = 16
mpl.rcParams["ytick.labelsize"] = 16

# Cleaner editable text in PDF/SVG
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"

# ============================================================
# Remake summary plots
# ============================================================
plot_alpha_vs_mean_slope(
    metric_key="mean_slope_W1",
    slopes_key="slopes_W1",
    metric_label="W1",
    filename="alpha_vs_mean_slope_W1.pdf",
)

plot_alpha_vs_mean_slope(
    metric_key="mean_slope_MMD",
    slopes_key="slopes_MMD",
    metric_label="MMD",
    filename="alpha_vs_mean_slope_MMD.pdf",
)

plot_alpha_vs_mean_slope(
    metric_key="mean_slope_W1",
    slopes_key="slopes_W1",
    metric_label="",
    filename="alpha_vs_mean_slope_W1_noLabel.pdf",
)

plot_alpha_vs_mean_slope(
    metric_key="mean_slope_MMD",
    slopes_key="slopes_MMD",
    metric_label="",
    filename="alpha_vs_mean_slope_MMD_noLabel.pdf",
)

print(f"\nDone. Remade figures are in:\n{REMAKE_FIG_DIR}")

### KDE - Slope Plots

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# ============================================================
# Update sizing
# ============================================================
import matplotlib as mpl 
mpl.rcParams["font.sans-serif"] = [
    "Arial",
]

# Huge font 
mpl.rcParams["font.size"] = 12
mpl.rcParams["axes.titlesize"] = 26
mpl.rcParams["axes.labelsize"] = 26
mpl.rcParams["legend.fontsize"] = 17 # was 15 
mpl.rcParams["legend.title_fontsize"] = 16
mpl.rcParams["xtick.labelsize"] = 24
mpl.rcParams["ytick.labelsize"] = 24

# Cleaner editable text in PDF/SVG
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"


# ============================================================
# Point this to the saved run folder
# ============================================================
RUN_DIR = Path("logs/KDE/alpha_h0=0.5_T=2000_m1=50_nreps=100_2026-05-19-14:01")
RESULTS_PATH = RUN_DIR / "all_results.json"

REMAKE_FIG_DIR = RUN_DIR / "remade_figures"
REMAKE_FIG_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# Reconstruct the true density grid from original constants
# ============================================================
w1 = 0.35
mu1, sigma1 = -2.0, 0.8
mu2, sigma2 = 1.0, 1.3

left = min(mu1 - 6 * sigma1, mu2 - 6 * sigma2)
right = max(mu1 + 6 * sigma1, mu2 + 6 * sigma2)
m_grid = 500

xg = np.linspace(left, right, m_grid)

f_true_g = (
    w1 * norm.pdf(xg, mu1, sigma1)
    + (1.0 - w1) * norm.pdf(xg, mu2, sigma2)
)


# ============================================================
# Read saved JSON
# ============================================================
with open(RESULTS_PATH, "r") as f:
    all_results = json.load(f)

print(f"Loaded {len(all_results)} saved result entries.")


# ============================================================
# Plot helpers
# ============================================================
def save_fig_both(base_path, dpi=150):
    """
    Save current matplotlib figure as both PNG and PDF.

    This version preserves decimal values in filenames, e.g.
    density_saved_p=0.50_alpha=0.50.png
    density_saved_p=0.50_alpha=0.50.pdf
    """
    base_path = Path(base_path)

    # Only strip suffix if the path explicitly ends in .png or .pdf.
    # This avoids chopping off decimal endings like alpha=0.50.
    if base_path.suffix.lower() in [".png", ".pdf"]:
        base_path = base_path.with_suffix("")

    png_path = Path(str(base_path) + ".png")
    pdf_path = Path(str(base_path) + ".pdf")

    plt.savefig(png_path, dpi=dpi, bbox_inches="tight")
    plt.savefig(pdf_path, bbox_inches="tight")

    print(f"Saved {png_path}")
    print(f"Saved {pdf_path}")


def plot_density_from_saved(res):
    """
    Remake density plot using the final saved f_hat_history
    from the saved result entry.
    
    Note: in your current script, all_results stores last_out,
    so this is the density from the LAST repetition, not the first repetition.
    """
    p = float(res["p_target"])
    alpha = float(res["alpha"])

    f_hat_history = np.asarray(res["f_hat_history"])
    f_hat_final = f_hat_history[-1]

    plt.figure(figsize=(7, 4))
    plt.plot(xg, f_true_g, linewidth=2, label="True density")
    plt.plot(xg, f_hat_final, linewidth=2, label="Estim. density")
    plt.xlabel("x")
    plt.ylabel("density")
    # plt.title(rf"Estimated density: p={p:.2f}, $\alpha$={alpha:.1f}")
    plt.grid(True, axis="y", ls="--", alpha=0.3)
    plt.legend()
    plt.tight_layout()

    out_path = REMAKE_FIG_DIR / f"density_saved_p={p:.2f}_alpha={alpha:.2f}"
    save_fig_both(out_path, dpi=150)
    plt.close()


def plot_with_trend(M, values, metric_name, p_val, alpha_val, drop_first=30, save_path=None):
    eps = 1e-20
    M = np.asarray(M, dtype=float)
    values = np.asarray(values, dtype=float)

    log_M = np.log10(M)
    log_val = np.log10(values + eps)

    plt.figure(figsize=(6, 4))
    plt.plot(log_M, log_val, "o-", alpha=0.5, markersize=3, label="Distr. Loss")

    if len(M) > drop_first + 5:
        x_fit = log_M[drop_first:]
        y_fit = log_val[drop_first:]
        slope, intercept = np.polyfit(x_fit, y_fit, 1)
        y_pred = slope * x_fit + intercept
        plt.plot(x_fit, y_pred, "r--", linewidth=2, label=f"Fit slope: {slope:.3f}")

    plt.xlabel(r"$\log_{10}$(M$_n$)")

    if metric_name == "W1":
        # plt.title(rf"$W_1$ Scaling: p={p_val:.2f}, $\alpha$={alpha_val:.2f}")
        plt.ylabel(r"$\log_{10}(W_1)$")
    else:
        # plt.title(rf"{metric_name} Scaling: p={p_val:.2f}, $\alpha$={alpha_val:.2f}")
        plt.ylabel(rf"$\log_{{10}}$({metric_name})")

    plt.legend()
    plt.grid(True, which="both", ls="--", alpha=0.3)
    plt.tight_layout()

    if save_path is not None:
        save_fig_both(save_path, dpi=150)
        plt.close()
    else:
        plt.show()


def remake_scaling_plots(res):
    p = float(res["p_target"])
    alpha = float(res["alpha"])

    M = np.asarray(res["M"], dtype=float)
    W1 = np.asarray(res["W1"], dtype=float)
    MMD2 = np.asarray(res["MMD2"], dtype=float)
    MMD = np.sqrt(np.maximum(MMD2, 0.0))

    run_label = f"p={p:.2f}_alpha={alpha:.2f}"

    plot_with_trend(
        M,
        W1,
        "W1",
        p,
        alpha,
        drop_first=30,
        save_path=REMAKE_FIG_DIR / f"{run_label}_W1",
    )

    plot_with_trend(
        M,
        MMD,
        "MMD",
        p,
        alpha,
        drop_first=30,
        save_path=REMAKE_FIG_DIR / f"{run_label}_MMD",
    )

    plot_with_trend(
        M,
        MMD2,
        "MMD^2",
        p,
        alpha,
        drop_first=30,
        save_path=REMAKE_FIG_DIR / f"{run_label}_MMD2",
    )


def plot_alpha_vs_mean_slope(metric_key, slopes_key, metric_label, filename, use_error_bars=False):
    unique_ps = sorted(set(float(res["p_target"]) for res in all_results))

    plt.figure(figsize=(6, 4))
    DARK_BLUE = "#244B99"
    RED = "#CC0000"

    for p in unique_ps:
        subset = [r for r in all_results if float(r["p_target"]) == p]

        alpha_plot = np.array([float(r["alpha"]) for r in subset])
        means = np.array([float(r[metric_key]) for r in subset])

        order = np.argsort(alpha_plot)
        alpha_plot = alpha_plot[order]
        means = means[order]

        vals_plot = -means

        if use_error_bars:
            stds = np.array([
                np.asarray(r[slopes_key], dtype=float).std(ddof=1)
                for r in subset
            ])
            stds = stds[order]

            n_reps = len(subset[0][slopes_key])
            err_plot = 1.96 * stds / np.sqrt(n_reps)

            plt.errorbar(
                alpha_plot,
                vals_plot,
                yerr=err_plot,
                fmt="o-",
                color=DARK_BLUE,
                ecolor=DARK_BLUE,
                elinewidth=1,
                capsize=3,
                label="Empirical",
            )
        else:
            plt.plot(alpha_plot, vals_plot, "o-", color=DARK_BLUE, label="Empirical")

        theo_vals = np.minimum(p, alpha_plot)
        plt.plot(alpha_plot, theo_vals, "--", color=RED, label="Theoretical")

    plt.xlabel(r"$\alpha$")

    if metric_label == "W1":
        plt.ylabel(r"Mean Slope $W_1$")
        # plt.title(r"Mean $W_1$ Slope vs $\alpha$")
    elif metric_label == 'MMD':
        plt.ylabel(f"Mean Slope {metric_label}")
        # plt.title(rf"Mean {metric_label} Slope vs $\alpha$")
    else: 
        pass # no label 

    plt.grid(True, axis="y", ls="--", alpha=0.3)
    plt.ylim(0.0, 0.65)
    plt.yticks(np.arange(0.0, 0.7, 0.1))

    handles, labels = plt.gca().get_legend_handles_labels()
    uniq = dict(zip(labels, handles))

    legend = False
    if legend:
        plt.legend(uniq.values(), uniq.keys())

    plt.tight_layout()

    out_path = REMAKE_FIG_DIR / filename
    save_fig_both(out_path, dpi=150)
    plt.close()


# ============================================================
# Remake all per-alpha plots
# ============================================================
for res in all_results:
    remake_scaling_plots(res)
    plot_density_from_saved(res)



### KDE - Summary Plots

In [ ]:
# ============================================================
# Update sizing
# ============================================================
import matplotlib as mpl 
mpl.rcParams["font.sans-serif"] = [
    "Arial",
    # "Helvetica",
    # "DejaVu Sans",
    # "Liberation Sans"
]

# enforce math to be arial
# mpl.rcParams["mathtext.default"] = "regular"

# smaller font 
mpl.rcParams["font.size"] = 12
mpl.rcParams["axes.titlesize"] = 18
mpl.rcParams["axes.labelsize"] = 18
mpl.rcParams["legend.fontsize"] = 11
mpl.rcParams["legend.title_fontsize"] = 14
# Axis tick label sizes
mpl.rcParams["xtick.labelsize"] = 16
mpl.rcParams["ytick.labelsize"] = 16

# Cleaner editable text in PDF/SVG
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
# ============================================================
# Remake summary plots
# ============================================================
plot_alpha_vs_mean_slope(
    metric_key="mean_slope_W1",
    slopes_key="slopes_W1",
    metric_label="W1",
    filename="alpha_vs_mean_slope_W1",
)

plot_alpha_vs_mean_slope(
    metric_key="mean_slope_MMD",
    slopes_key="slopes_MMD",
    metric_label="MMD",
    filename="alpha_vs_mean_slope_MMD",
)

plot_alpha_vs_mean_slope(
    metric_key="mean_slope_W1",
    slopes_key="slopes_W1",
    metric_label="",
    filename="alpha_vs_mean_slope_W1_noLabel",
)

plot_alpha_vs_mean_slope(
    metric_key="mean_slope_MMD",
    slopes_key="slopes_MMD",
    metric_label="",
    filename="alpha_vs_mean_slope_MMD_noLabel",
)

print(f"\nDone. Remade figures are in:\n{REMAKE_FIG_DIR}")